## Deployments

In this notebook, we look at Kubernetes Deployments.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool on your computer.

## Create a Deployment

- The Deployment definition is in the YAML file `manifests/deploy-example.yaml`

In [1]:
!kubectl apply -f manifests/deploy-example.yaml

deployment.apps/deploy-example created


## Let's look at the Deployment's YAML

**Note:**

- The number of `replicas` is set to 3.
- The Pod template's container is listening on `containerPort` 80.
- The Pod template's `labels` are `app: nginx` and `type: prod`
  - The Deployment's `matchLabels` match these labels
    - Therefore, the Deployment will create 3 replicas from the Pod template.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-example           # the deployment's name is deploy-example
spec:
  replicas: 3                    # the deployment defines three replicas
  revisionHistoryLimit: 3        # the deployment defines a maximum of three revisions in the history list
  selector:
    matchLabels:
      app: nginx                 # the deployment has two matchLabels that match the Pod template's labels below
      type: prod                 # app = nginx and type = prod
  template:
    metadata:
      name: myapp-pod
      labels:
        app: nginx               # the Pod template has two labels defined:
        type: prod               # app = nginx and type = prod
    spec:
      containers:
      - name: nginx-container    # the container's name is nginx-container
        image: nginx-container   # the container is based on the nginx:alpine image
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi    
        ports:
        - containerPort: 80      # the Pod template's container is listening on containerPort 80
```

In [2]:
#!type manifests\deploy-example.yaml # use this on Windows
!cat manifests/deploy-example.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-example
spec:
  replicas: 3
  revisionHistoryLimit: 3
  selector:
    matchLabels:
      app: nginx
      env: prod
  template:
    metadata:
      labels:
        app: nginx
        env: prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi        
        ports:
        - containerPort: 80

## List Deployments
- Note that `deployments` can be abbreviated to `deploy`.

In [3]:
#!kubectl get deploy -o wide
!kubectl get deployments -o wide

NAME             READY   UP-TO-DATE   AVAILABLE   AGE   CONTAINERS   IMAGES         SELECTOR
deploy-example   3/3     3            3           11s   nginx        nginx:alpine   app=nginx,env=prod


## Get detailed information about the Deployment

- Notice `Replicas` tells us there are 3 desired, 3 updated, 3 total and 3 available Pod replicas.

In [4]:
#!kubectl describe deploy deploy-example
!kubectl describe deployment deploy-example

Name:                   deploy-example
Namespace:              default
CreationTimestamp:      Sat, 14 Mar 2026 20:50:53 +0100
Labels:                 <none>
Annotations:            deployment.kubernetes.io/revision: 1
Selector:               app=nginx,env=prod
Replicas:               3 desired | 3 updated | 3 total | 3 available | 0 unavailable
StrategyType:           RollingUpdate
MinReadySeconds:        0
RollingUpdateStrategy:  25% max unavailable, 25% max surge
Pod Template:
  Labels:  app=nginx
           env=prod
  Containers:
   nginx:
    Image:      nginx:alpine
    Port:       80/TCP
    Host Port:  0/TCP
    Limits:
      cpu:     250m
      memory:  256Mi
    Requests:
      cpu:         100m
      memory:      128Mi
    Environment:   <none>
    Mounts:        <none>
  Volumes:         <none>
  Node-Selectors:  <none>
  Tolerations:     <none>
Conditions:
  Type           Status  Reason
  ----           ------  ------
  Available      True    MinimumReplicasAvailable
  Pr

## List ReplicaSets

- Notice the name of the ReplicaSet consists of the Deployment name `deploy-example` and a randomly generated suffix.

In [5]:
#!kubectl get rs -o wide
!kubectl get replicasets -o wide

NAME                       DESIRED   CURRENT   READY   AGE   CONTAINERS   IMAGES         SELECTOR
deploy-example-6f689b949   3         3         3       19s   nginx        nginx:alpine   app=nginx,env=prod,pod-template-hash=6f689b949


## Get detailed information about all ReplicaSets

- Notice that we get detailed information about all ReplicaSets if we don't explicitly specify a replicaset.

In [6]:
#!kubectl describe rs
!kubectl describe replicasets

Name:           deploy-example-6f689b949
Namespace:      default
Selector:       app=nginx,env=prod,pod-template-hash=6f689b949
Labels:         app=nginx
                env=prod
                pod-template-hash=6f689b949
Annotations:    deployment.kubernetes.io/desired-replicas: 3
                deployment.kubernetes.io/max-replicas: 4
                deployment.kubernetes.io/revision: 1
Controlled By:  Deployment/deploy-example
Replicas:       3 current / 3 desired
Pods Status:    3 Running / 0 Waiting / 0 Succeeded / 0 Failed
Pod Template:
  Labels:  app=nginx
           env=prod
           pod-template-hash=6f689b949
  Containers:
   nginx:
    Image:      nginx:alpine
    Port:       80/TCP
    Host Port:  0/TCP
    Limits:
      cpu:     250m
      memory:  256Mi
    Requests:
      cpu:         100m
      memory:      128Mi
    Environment:   <none>
    Mounts:        <none>
  Volumes:         <none>
  Node-Selectors:  <none>
  Tolerations:     <none>
Events:
  Type    Reason 

## List Pods

- Notice the name of the Pods consists of:
  - The Deployment name `deploy-example`
  - Followed by the ReplicaSet suffix.
  - And finally a randomly generated suffix.

In [7]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                             READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
deploy-example-6f689b949-297wc   1/1     Running   0          25s   10.244.0.12   minikube   <none>           <none>
deploy-example-6f689b949-949b8   1/1     Running   0          25s   10.244.0.10   minikube   <none>           <none>
deploy-example-6f689b949-q8qn9   1/1     Running   0          25s   10.244.0.11   minikube   <none>           <none>


## Get detailed information about the Pods

- Notice the name of the deployment is used here, so all three pods are described.

In [8]:
#!kubectl describe po deploy-example
!kubectl describe pod deploy-example

Name:             deploy-example-6f689b949-297wc
Namespace:        default
Priority:         0
Service Account:  default
Node:             minikube/192.168.49.2
Start Time:       Sat, 14 Mar 2026 20:50:54 +0100
Labels:           app=nginx
                  env=prod
                  pod-template-hash=6f689b949
Annotations:      <none>
Status:           Running
IP:               10.244.0.12
IPs:
  IP:           10.244.0.12
Controlled By:  ReplicaSet/deploy-example-6f689b949
Containers:
  nginx:
    Container ID:   docker://6638898bfad1895a8e31f0fc400ea0fd9d8dfa7e6cdb290a461a6bcd6a3fa8f2
    Image:          nginx:alpine
    Image ID:       docker-pullable://nginx@sha256:f46cb72c7df02710e693e863a983ac42f6a9579058a59a35f1ae36c9958e4ce0
    Port:           80/TCP
    Host Port:      0/TCP
    State:          Running
      Started:      Sat, 14 Mar 2026 20:50:55 +0100
    Ready:          True
    Restart Count:  0
    Limits:
      cpu:     250m
      memory:  256Mi
    Requests:
      cpu: 

## Delete the Deployment

In [9]:
!kubectl delete -f manifests/deploy-example.yaml

deployment.apps "deploy-example" deleted from default namespace


## List Deployments, ReplicaSets and Pods

- Notice that deleting the Deployment also deleted the ReplicaSet and the Pods defined in the Deployment.

In [10]:
!kubectl get deploy
!kubectl get rs
!kubectl get po
#!kubectl get deployments
#!kubectl get replicasets
#!kubectl get pods

No resources found in default namespace.
No resources found in default namespace.
No resources found in default namespace.
